# 02 · Fine-tune pipeline (run all)

End-to-end: data prep (structures + ColabFold MSAs) → preflight → fine-tune → evaluate baseline vs fine-tuned.

**Script equivalent:** `scripts/run_all.sh`.

> **Runs on any GPU notebook platform** — Google Colab, Kaggle, Paperspace Gradient, SageMaker Studio Lab, Lightning AI. Make sure a **GPU runtime** is selected before running.

> Each notebook mirrors a shell script in `scripts/`; you can use either form.

> **GPU size matters.** A free **T4 (16 GB)** can only run the *smoke* profile (`GPU_PROFILE='small'`). A real fine-tune needs an **A100/H100**.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo 'No GPU! Enable a GPU runtime.'

## Install OpenFold3 + weights
Uses the pip install path (simplest across cloud platforms). The `setup_openfold` step is interactive, so we feed answers non-interactively: default cache, default dir, `1` (default checkpoint), `no` (skip the heavy integration tests).

In [ ]:
!pip install -q "openfold3[cuequivariance]"
!printf '\n\n1\nno\n' | setup_openfold

## Get the kit

In [ ]:
!git clone -q https://github.com/recep2244/openfold3-finetune-kit.git || true
%cd openfold3-finetune-kit
!chmod +x scripts/*.sh

## Parameters
Edit the two ID lists for your target. TRAIN = model learns from; VAL = held-out (never seen in training; used to measure improvement).

In [ ]:
TRAIN_IDS = "5SDY 5SIQ 5SI7 5SIG 5SI5 5SI8 5SIY 5SG5 5SGL 5SIH"
VAL_IDS   = "5SH0 5SE0 5SHR 5SJL 5SH8 5SF4 5SFG 5SE5 5SHK 5SEE"
GPU_PROFILE = "small"   # "small" for T4/16GB smoke test, "big" for A100/H100
import os
os.environ.update(TRAIN_IDS=TRAIN_IDS, VAL_IDS=VAL_IDS, GPU_PROFILE=GPU_PROFILE)

In [ ]:
import os, pathlib, openfold3
# pip install (no ~/openfold-3 clone): point OF3_REPO at the installed package root
pkg = pathlib.Path(openfold3.__file__).resolve().parent.parent
os.environ["OF3_REPO"] = str(pkg)
os.environ["WORK"] = "/content/openfold3_run" if pathlib.Path("/content").exists() else os.path.abspath("openfold3_run")
os.environ["CKPT"] = os.path.expanduser("~/.openfold3/of3-p2-155k.pt")
print("OF3_REPO =", os.environ["OF3_REPO"]); print("WORK =", os.environ["WORK"])

## Run the whole pipeline
On a T4 this validates the pipeline end-to-end; on an A100 it runs the real fine-tune (can take hours). It stops with a clear message if any stage fails.

In [ ]:
!bash scripts/run_all.sh

## Persist results (optional)
Cloud disks are temporary. On Colab this saves to Google Drive; on other platforms use the platform's own persistent storage or download `\$WORK/eval/out/results.csv`.

In [ ]:
import os, shutil, pathlib
try:
    from google.colab import drive
    drive.mount('/content/drive')
    dst = '/content/drive/MyDrive/openfold3_run'
    os.makedirs(dst, exist_ok=True)
    src = os.environ['WORK']
    for sub in ('train_out/checkpoints', 'eval/out'):
        p = pathlib.Path(src, sub)
        if p.exists():
            shutil.copytree(p, pathlib.Path(dst, sub), dirs_exist_ok=True)
    print('saved to', dst)
except ImportError:
    print('Not on Colab — download', os.environ['WORK'], 'via your platform tools.')